In [322]:
import cuml
import cudf
import cupy
%load_ext cuml.accel

The cuml.accel extension is already loaded. To reload it, use:
  %reload_ext cuml.accel


In [323]:
%reload_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
from sklearn import set_config
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
import xgboost as xgb
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, MinMaxScaler
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import classification_report
from sklearn_extra import GroupImputer
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
import optuna

In [324]:
train_df = pd.read_csv("./data/train.csv")
test_df = pd.read_csv("./data/test.csv")

In [325]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [326]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    object 
 3   Sex          418 non-null    object 
 4   Age          332 non-null    float64
 5   SibSp        418 non-null    int64  
 6   Parch        418 non-null    int64  
 7   Ticket       418 non-null    object 
 8   Fare         417 non-null    float64
 9   Cabin        91 non-null     object 
 10  Embarked     418 non-null    object 
dtypes: float64(2), int64(4), object(5)
memory usage: 36.1+ KB


In [327]:

def add_family_size(df):
    # df = df.copy()
    df["FamilySize"] = df["SibSp"] + df["Parch"]
    return df

def strip_name(df):
    df["Name"] = df["Name"].str.strip()
    return df

def extract_title(df):
    # df = df.copy()
    df["Title"] = df["Name"].str.extract(r"([a-zA-Z]+)\.")
    main_title = ["Mr", "Mrs", "Miss", "Master"]
    df["Title"] = df["Title"].apply(lambda x: x if x in main_title else "Rare")
    
    return df

# def age_binning(df):
#     # df = df.copy()
#     df["Age"] = pd.qcut(df["Age"], q=5, labels=False)
    
#     return df

# def fare_binning(df):
#     # df = df.copy()
#     df["Fare"] = pd.qcut(df["Fare"], q=5, labels=False)
    
#     return df


def extract_deck(df):
    # df = df.copy()
    df["Deck"] = df["Cabin"].str.extract(r"([a-zA-Z])")
    deck_grouping = {
        'A': 'ABC',
        'B': 'ABC',
        'C': 'ABC',
        'D': 'DE',
        'E': 'DE',
        'F': 'FGTM',
        'G': 'FGTM',
        'T': 'FGTM', # Assigning rare decks to a similar group
        'M': 'FGTM'
        # Any key not listed will result in NaN after mapping unless handled
    }
    
    df["Deck"] = df["Deck"].map(deck_grouping).fillna("M")
    
    return df
    

In [328]:
X_train, y_train = train_df.drop("Survived", axis=1), train_df["Survived"]
X_test = test_df

In [ ]:
set_config(transform_output="default")

embarked_transformer = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown='ignore') # Good practice for OHE
)

preprocess_pipeline = make_pipeline(
    (GroupImputer(["Pclass", "Sex"], "Age")),
    (GroupImputer(["Pclass", "Sex"], "Fare")),
    (GroupImputer(["Ticket"], "Cabin", strategy="most_common")),
    
    (FunctionTransformer(add_family_size, validate=False)),
    (FunctionTransformer(strip_name, validate=False)),
    (FunctionTransformer(extract_title, validate=False)),
    (FunctionTransformer(extract_deck, validate=False)),
    # (FunctionTransformer(age_binning, validate=False)),
    # (FunctionTransformer(fare_binning, validate=False)),
    (make_column_transformer(
        # (OrdinalEncoder(), ["Sex", "Deck", "Pclass"]),
        (embarked_transformer, ["Embarked"]),
        (OneHotEncoder(), ["Sex", "Deck", "Pclass", "Title"]),
        (MinMaxScaler(), ["Age", "Fare", "SibSp", "Parch", "FamilySize"]),
        remainder='drop',
        verbose_feature_names_out=False
    ))
)



# pipeline.fit(train_df)

In [330]:
processed_train_df = preprocess_pipeline.fit_transform(X_train)
processed_test_df = preprocess_pipeline.transform(X_test)

In [331]:
preprocess_pipeline.steps[-1][1].get_feature_names_out()

array(['Embarked_C', 'Embarked_Q', 'Embarked_S', 'Sex_female', 'Sex_male',
       'Deck_ABC', 'Deck_DE', 'Deck_FGTM', 'Pclass_1', 'Pclass_2',
       'Pclass_3', 'Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs',
       'Title_Rare', 'Age', 'Fare', 'SibSp', 'Parch', 'FamilySize'],
      dtype=object)

In [332]:
# processed_train_df = pd.DataFrame(processed_train_df, columns=preprocess_pipeline.steps[-1][1].get_feature_names_out())
# process_test_df = pd.DataFrame(process_test_df, columns=preprocess_pipeline.steps[-1][1].get_feature_names_out())

In [333]:
# processed_train_df_gpu = cupy.asarray(processed_train_df)
# y_gpu = cudf.from_pandas(y_train)

In [334]:

# def objective(trial):
#     params = {
#         'n_estimators': trial.suggest_int('n_estimators', 50, 300),
#         'max_depth': trial.suggest_int('max_depth', 3, 10),
#         'max_leaves': trial.suggest_int('max_leaves', 16, 256),
#         'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
#         'subsample': trial.suggest_float('subsample', 0.6, 1.0),
#         'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
#         'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
#         'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
#         'tree_method': 'hist',
#         'device': 'cuda',
#         'random_state': 42,
#     }
    
#     model = xgb.XGBClassifier(**params)
#     score = cross_val_score(model, processed_train_df, y_train, cv=3, scoring="f1").mean()
    
#     return score

# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=200, show_progress_bar=False)

# print(f"Best score: {study.best_value:4f}")
# print(f"Best params: {study.best_params}")

In [335]:
clean_params = study.trials_dataframe().sort_values(by="value", ascending=False).iloc[1].to_dict()

In [336]:
clean_params

{'number': 126,
 'value': 0.8552068294520119,
 'datetime_start': Timestamp('2025-12-24 23:54:44.427931'),
 'datetime_complete': Timestamp('2025-12-24 23:54:52.643784'),
 'duration': Timedelta('0 days 00:00:08.215853'),
 'params_lr_c': 0.0038389153626314627,
 'params_rf_criterion': 'gini',
 'params_rf_max_depth': 7,
 'params_rf_min_samples_leaf': 9,
 'params_rf_min_samples_split': 7,
 'params_rf_n_estimators': 409,
 'params_svc_c': 7.555455633227737,
 'params_svc_gamma': 'scale',
 'params_svc_kernel': 'linear',
 'params_w_lr': 0.24041189665334178,
 'params_w_rf': 0.16533409971613144,
 'params_w_svc': 0.5968092890480398,
 'params_w_xgb': 0.39566816978461794,
 'params_xgb_colsample_bytree': 0.9059120415214026,
 'params_xgb_learning_rate': 0.24454589141040348,
 'params_xgb_max_depth': 10,
 'params_xgb_n_estimators': 319,
 'params_xgb_subsample': 0.808492437966553,
 'state': 'COMPLETE'}

In [337]:
clean_params = {
    k.replace("params_", ""): v 
    for k, v in clean_params.items() 
    if k.startswith("params_")
}

In [338]:
# clean_params

In [339]:
param = {'n_estimators': 256,
 'max_depth': 5,
 'max_leaves': 128,
 'learning_rate': 0.1,
 'subsample': 0.9154939464334865,
 'reg_alpha': 0.3384165449961178,
 'reg_lambda': 1.3089360135857229,
 'min_child_weight': 5}

# # param = clean_params

train_pipeline = make_pipeline(
    (VotingClassifier([
        ("xgb",xgb.XGBClassifier(**param)), 
        ("rf",RandomForestClassifier(criterion='gini', 
            n_estimators=300,
            max_depth=5,
            min_samples_split=4,
            min_samples_leaf=5,
            oob_score=True,
            random_state=42,
            n_jobs=-1)),
        ("svc", SVC(C=5, probability=True)), 
        ("l_reg", LogisticRegression())
        ],
        voting="soft"
        )
    )
)


In [340]:
# Get cross-validated predictions


# y_pred_cv = cross_val_predict(train_pipeline, processed_train_df, y_train, cv=5)
# # Print the classification report
# print(classification_report(y_train, y_pred_cv, digits=4))

In [341]:
# from sklearn.model_selection import cross_validate

# out = cross_validate(train_pipeline, processed_train_df, y_train, cv=8, scoring=["accuracy", "f1"], return_train_score=True)

In [342]:
# out

In [343]:
# print(out["train_f1"].mean())
# print(out["train_f1"].std())
# print(out["test_f1"].mean())
# print(out["test_f1"].std())

In [344]:
# train_pipeline.fit(processed_train_df, y_train)

In [345]:
# y_pred = train_pipeline.predict(processed_test_df)

In [346]:
# train_pipeline.steps[0][1]

In [347]:
# import matplotlib.pyplot as plt

# # Create feature importance DataFrame
# feature_importance = pd.DataFrame({
#     'feature': preprocess_pipeline.steps[-1][1].get_feature_names_out(),
#     'importance': train_pipeline.steps[0][1].feature_importances_
# }).sort_values('importance', ascending=True)

# # Plot
# fig, ax = plt.subplots(figsize=(10, 6))
# bars = ax.barh(feature_importance['feature'], feature_importance['importance'])
# ax.bar_label(bars, fmt='%.3f', padding=3)
# ax.set_xlabel('Importance')
# ax.set_ylabel('Feature')
# ax.set_title('XGBoost Feature Importance')
# plt.tight_layout()
# plt.show()

In [348]:
gt = pd.read_csv("./out/gt.csv")

In [349]:
preprocess_pipeline.steps[-1][1].get_feature_names_out()

array(['Embarked_C', 'Embarked_Q', 'Embarked_S', 'Sex_female', 'Sex_male',
       'Deck_ABC', 'Deck_DE', 'Deck_FGTM', 'Pclass_1', 'Pclass_2',
       'Pclass_3', 'Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs',
       'Title_Rare', 'Age', 'Fare', 'SibSp', 'Parch', 'FamilySize'],
      dtype=object)

In [350]:
# print(classification_report(gt["Survived"], y_pred, digits=4))

In [351]:
def objective(trial):
    # --- 1. Define Search Space for XGBoost ---
    xgb_params = {
        'n_estimators': trial.suggest_int('xgb_n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('xgb_max_depth', 3, 10),
        'learning_rate': trial.suggest_float('xgb_learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('xgb_subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('xgb_colsample_bytree', 0.5, 1.0),
        'random_state': 42,
        'n_jobs': 1 # Set to 1 to avoid thread contention with Optuna
    }
    
    # --- 2. Define Search Space for Random Forest ---
    rf_params = {
        'n_estimators': trial.suggest_int('rf_n_estimators', 100, 500),
        'max_depth': trial.suggest_int('rf_max_depth', 5, 30),
        'min_samples_split': trial.suggest_int('rf_min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('rf_min_samples_leaf', 1, 10),
        'criterion': trial.suggest_categorical('rf_criterion', ['gini', 'entropy']),
        'random_state': 42,
        'n_jobs': 1
    }
    
    # --- 3. Define Search Space for SVC ---
    # SVC requires probability=True for soft voting
    svc_params = {
        'C': trial.suggest_float('svc_c', 1e-3, 10, log=True),
        'kernel': trial.suggest_categorical('svc_kernel', ['linear', 'rbf', 'poly']),
        'gamma': trial.suggest_categorical('svc_gamma', ['scale', 'auto']),
        'probability': True,
        'random_state': 42
    }
    
    # --- 4. Define Search Space for Logistic Regression ---
    lr_params = {
        'C': trial.suggest_float('lr_c', 1e-3, 10, log=True),
        'solver': 'lbfgs', # Fixed solver for simplicity
        'random_state': 42
    }

    # --- 5. Instantiate the Classifiers ---
    clf_xgb = xgb.XGBClassifier(**xgb_params)
    clf_rf = RandomForestClassifier(**rf_params)
    clf_svc = SVC(**svc_params)
    clf_lr = LogisticRegression(**lr_params)

    # --- 6. Define Voting Weights ---
    # We suggest a float for each model's weight
    w_xgb = trial.suggest_float("w_xgb", 0.0, 1.0)
    w_rf = trial.suggest_float("w_rf", 0.0, 1.0)
    w_svc = trial.suggest_float("w_svc", 0.0, 1.0)
    w_lr = trial.suggest_float("w_lr", 0.0, 1.0)
    
    # --- 7. Build the Voting Classifier ---
    voting_clf = VotingClassifier(
        estimators=[
            ('xgb', clf_xgb),
            ('rf', clf_rf),
            ('svc', clf_svc),
            ('l_reg', clf_lr)
        ],
        voting='soft',
        weights=[w_xgb, w_rf, w_svc, w_lr]
    )

    # --- 8. Create the Pipeline ---
    # Always recommended to scale data when using SVC/LogisticRegression
    pipeline = make_pipeline(voting_clf)
    
    # We use 'accuracy' here, but you can change to 'roc_auc', 'f1', etc.
    scores = cross_val_score(pipeline, processed_train_df, y_train, cv=5, scoring='accuracy', n_jobs=-1)
    
    return scores.mean()

In [352]:
# # Create the study object. 
# # direction='maximize' because we want higher accuracy.
# study = optuna.create_study(direction="maximize")

# # Run the optimization
# # n_trials depends on your time budget; 50-100 is a good start for this complexity.
# study.optimize(objective, n_trials=200)

# print("Best trial:")
# print(f"  Value: {study.best_value}")
# print("  Params: ")
# for key, value in study.best_params.items():
#     print(f"    {key}: {value}")

In [353]:
# study.best_params

In [368]:
def get_best_pipeline(best_params):
    # Extract params for XGB
    p_xgb = {
        'n_estimators': best_params['xgb_n_estimators'],
        'max_depth': best_params['xgb_max_depth'],
        'learning_rate': best_params['xgb_learning_rate'],
        'subsample': best_params['xgb_subsample'],
        'colsample_bytree': best_params['xgb_colsample_bytree'],
        'random_state': 42,
        'n_jobs': -1 
    }
    
    # Extract params for RF
    p_rf = {
        'n_estimators': best_params['rf_n_estimators'],
        'max_depth': best_params['rf_max_depth'],
        'min_samples_split': best_params['rf_min_samples_split'],
        'min_samples_leaf': best_params['rf_min_samples_leaf'],
        'criterion': best_params['rf_criterion'],
        'random_state': 42,
        'n_jobs': -1
    }
    
    # Extract params for SVC
    p_svc = {
        'C': best_params['svc_c'],
        'kernel': best_params['svc_kernel'],
        'gamma': best_params['svc_gamma'],
        'probability': True,
        'random_state': 42
    }
    
    # Extract params for Logistic Regression
    p_lr = {
        'C': best_params['lr_c'],
        'solver': 'lbfgs',
        'random_state': 42
    }
    
    # Extract Weights
    weights = [
        best_params['w_xgb'],
        best_params['w_rf'],
        best_params['w_svc'],
        best_params['w_lr']
    ]

    # Rebuild
    clf_xgb = xgb.XGBClassifier(**p_xgb)
    clf_rf = RandomForestClassifier(**p_rf)
    clf_svc = SVC(**p_svc)
    clf_lr = LogisticRegression(**p_lr)
    
    voting_clf = VotingClassifier(
        estimators=[('xgb', clf_xgb), ('rf', clf_rf), ('svc', clf_svc), ('l_reg', clf_lr)],
        voting='soft',
        weights=weights
    )
    
    return make_pipeline( voting_clf)

best_param = {'xgb_n_estimators': 290,
 'xgb_max_depth': 10,
 'xgb_learning_rate': 0.2512801201323504,
 'xgb_subsample': 0.7802108269027506,
 'xgb_colsample_bytree': 0.9520474770545232,
 'rf_n_estimators': 387,
 'rf_max_depth': 7,
 'rf_min_samples_split': 7,
 'rf_min_samples_leaf': 8,
 'rf_criterion': 'gini',
 'svc_c': 3.079132191483121,
 'svc_kernel': 'linear',
 'svc_gamma': 'scale',
 'lr_c': 0.0015202515812169594,
 'w_xgb': 0.3425891402508971,
 'w_rf': 0.16713294730947947,
 'w_svc': 0.5359566418155862,
 'w_lr': 0.21666595002876604}

# Get the best model
final_pipeline = get_best_pipeline(best_param)

# Fit on the full dataset
final_pipeline.fit(processed_train_df, y_train)

[2025-12-25 00:37:56.490] [CUML] [warning] Random state is currently ignored by probabilistic SVC


,steps,"[('votingclassifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,estimators,"[('xgb', ...), ('rf', ...), ...]"
,voting,'soft'
,weights,"[0.3425891402508971, 0.16713294730947947, ...]"
,n_jobs,None
,flatten_transform,True
,verbose,False
,objective,'binary:logistic'


In [369]:
y_pred = final_pipeline.predict(processed_test_df)

In [370]:
print(classification_report(gt["Survived"], y_pred, digits=4))

              precision    recall  f1-score   support

           0     0.7985    0.8385    0.8180       260
           1     0.7103    0.6519    0.6799       158

    accuracy                         0.7679       418
   macro avg     0.7544    0.7452    0.7489       418
weighted avg     0.7652    0.7679    0.7658       418



In [371]:
from sklearn.model_selection import cross_validate

out = cross_validate(train_pipeline, processed_train_df, y_train, cv=5, scoring=["accuracy", "f1"], return_train_score=True)

In [372]:
print(out["test_f1"].mean())
print(out["train_f1"].mean())
out

0.7699865732194652
0.8239797256509116


{'fit_time': array([0.46426034, 0.44881725, 0.59255791, 0.49825954, 0.60263491]),
 'score_time': array([0.05960083, 0.0652411 , 0.07181144, 0.05746222, 0.06458449]),
 'test_accuracy': array([0.82681564, 0.83146067, 0.87640449, 0.80898876, 0.84831461]),
 'train_accuracy': array([0.87640449, 0.87798036, 0.87517532, 0.86816269, 0.87237027]),
 'test_f1': array([0.75968992, 0.76190476, 0.82258065, 0.71186441, 0.79389313]),
 'train_f1': array([0.82677165, 0.82840237, 0.82851638, 0.81640625, 0.81980198])}

In [373]:
out_val = cross_validate(final_pipeline, processed_train_df, y_train, cv=5, scoring=["accuracy", "f1"], return_train_score=True)

[2025-12-25 00:38:02.081] [CUML] [warning] Random state is currently ignored by probabilistic SVC
[2025-12-25 00:38:03.173] [CUML] [warning] Random state is currently ignored by probabilistic SVC
[2025-12-25 00:38:04.267] [CUML] [warning] Random state is currently ignored by probabilistic SVC
[2025-12-25 00:38:05.326] [CUML] [warning] Random state is currently ignored by probabilistic SVC
[2025-12-25 00:38:06.751] [CUML] [warning] Random state is currently ignored by probabilistic SVC


In [374]:
print(out_val["test_f1"].mean())
print(out_val["train_f1"].mean())
out_val

0.8004612422296562
0.8499808121113448


{'fit_time': array([1.0372777 , 1.06650305, 1.00721359, 1.10695219, 1.26737618]),
 'score_time': array([0.03651285, 0.0384171 , 0.03872275, 0.03788328, 0.04149723]),
 'test_accuracy': array([0.87150838, 0.84269663, 0.87078652, 0.8258427 , 0.87640449]),
 'train_accuracy': array([0.90308989, 0.90182328, 0.88639551, 0.89060309, 0.88779804]),
 'test_f1': array([0.82442748, 0.78125   , 0.81889764, 0.7394958 , 0.83823529]),
 'train_f1': array([0.86282306, 0.86111111, 0.83960396, 0.84765625, 0.83870968])}